In [1]:
!pip install biopython pandas numpy scikit-learn seaborn

from collections import Counter
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# ================================
# 0. Constant Definitions
# ================================

AAS = "ACDEFGHIKLMNPQRSTVWY"
SW_WINDOWS = [20, 40]

AA_GROUPS = {
    "KR": "KR",
    "KRH": "KRH",
    "ED": "ED",
}

AA_GROUPS_EXT = {
    "STNQCH": "STNQCH",
    "ILMV": "ILMV",
    "FWY": "FWY",
}

# Merge extended and charge group definitions for AAC calculation
AAC_GROUPS_ALL = {**AA_GROUPS_EXT, **AA_GROUPS}


def clean_seq(seq):
    """Replaces non-standard amino acid U (Selenocysteine) with C (Cysteine)."""
    return seq.replace("U", "C")


# ================================
# 1. Load Data IDs (High / Mid / Low)
# ================================

df_high = pd.read_excel("High_group_ID.xlsx")
df_mid = pd.read_excel("Mid_group_ID.xlsx")
df_low = pd.read_excel("Low_group_ID.xlsx")

df_high["Class3"] = "High"
df_mid["Class3"] = "Mid"
df_low["Class3"] = "Low"

df_class = pd.concat([df_high, df_mid, df_low], ignore_index=True)

# ================================
# 2. Sequence Data
# ================================

seqs = {
    rec.id: str(rec.seq)
    for rec in SeqIO.parse("ALL_protein_sequences.fasta", "fasta")
}

df_seq = pd.DataFrame(seqs.items(), columns=["ID", "Sequence"])

# Merge class data directly with sequence data without NetSurfP features
df = df_class.merge(df_seq, on="ID")
df["Sequence"] = df["Sequence"].map(clean_seq)

df = df.reset_index(drop=True)

# ================================
# 3. Binary Class Labels (High / Low)
# ================================

df_binary = df[df["Class3"].isin(["High", "Low"])].copy()
df_binary["Class2"] = df_binary["Class3"].map({"Low": 0, "High": 1})

# ================================
# 4. Physicochemical Properties + AAC (+ Group AAC)
# ================================


def physchem_features(seq):
    prot = ProteinAnalysis(seq)
    aa = prot.count_amino_acids()
    L = len(seq)

    # --- Basic physicochemical properties ---
    base = [
        prot.molecular_weight(),
        prot.gravy(),
        prot.charge_at_pH(7.5),
        prot.charge_at_pH(6.8),
        prot.charge_at_pH(6.8) - prot.charge_at_pH(7.5),
        prot.isoelectric_point(),
    ]

    # --- Single Amino Acid Composition (AAC) ---
    aac = [aa[a] / L for a in AAS]

    # --- Grouped Amino Acid Composition (EXT + charge) ---
    aac_group = [
        sum(aa[a] for a in group) / L for group in AAC_GROUPS_ALL.values()
    ]

    return base + aac + aac_group


pc_cols = (
    ["MW", "GRAVY", "Charge_7.5", "Charge_6.8", "DeltaCharge_6.8_7.5", "pI"]
    + [f"AAC_{a}" for a in AAS]
    + [f"AACgroup_{k}" for k in AAC_GROUPS_ALL]
)

df_pc_all = pd.DataFrame(
    [physchem_features(s) for s in df["Sequence"]], columns=pc_cols
)

# ================================
# 5. Terminal Positional Bias Index
# ================================


def termial_bias(seq, aa_set):
    """Calculates the mean relative positional bias from the sequence center."""
    L = len(seq)
    positions = [
        abs((i + 0.5) / L - 0.5) for i, a in enumerate(seq) if a in aa_set
    ]
    return np.mean(positions) if positions else 0.0


df_term_aa = pd.DataFrame(
    [[termial_bias(s, a) for a in AAS] for s in df["Sequence"]],
    columns=[f"TermBias_{a}" for a in AAS],
)

df_term_group = pd.DataFrame(
    [
        [termial_bias(s, g) for g in AA_GROUPS_EXT.values()]
        for s in df["Sequence"]
    ],
    columns=[f"TermBias_{k}" for k in AA_GROUPS_EXT],
)

df_term_charge = pd.DataFrame(
    [[termial_bias(s, g) for g in AA_GROUPS.values()] for s in df["Sequence"]],
    columns=[f"TermBias_{k}" for k in AA_GROUPS],
)


# ================================
# 6. Sliding Window Variance Normalized by Theoretical Expectation
# ================================


def sliding_window_var_norm_aa(seq, aas):
    """Calculates normalized variance of single amino acid frequencies across sliding windows."""
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for a in aas:
            p = cnt_all[a] / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i : i + w]
                vals.append(sub.count(a) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out


df_sw_aa = pd.DataFrame(
    [sliding_window_var_norm_aa(s, AAS) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{a}" for w in SW_WINDOWS for a in AAS],
)


def sliding_window_var_norm_group(seq, groups):
    """Calculates normalized variance of amino acid group frequencies across sliding windows."""
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for k, g in groups.items():
            p = sum(cnt_all[a] for a in g) / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i : i + w]
                vals.append(sum(sub.count(a) for a in g) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out


df_sw_group = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS_EXT) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS_EXT],
)

df_sw_charge = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS],
)

# ================================
# 7. k-mer (k=2): Symmetrization & Mean Frequency Filtering
# ================================

KMER_K = 2

# Symmetrize pair frequencies (e.g., combine AB and BA into AA, AC, AD, ..., YY)
KMER2_SYM_LIST = sorted(
    set("".join(sorted(a + b)) for a in AAS for b in AAS)
)


def kmer2_symmetric_features(seq):
    L = len(seq)

    counts = Counter(
        "".join(sorted(seq[i : i + 2])) for i in range(L - 1)
    )

    denom = max(L - 1, 1)
    return [counts.get(km, 0) / denom for km in KMER2_SYM_LIST]


df_kmer2_sym = pd.DataFrame(
    [kmer2_symmetric_features(s) for s in df["Sequence"]],
    columns=[f"Pair_{km}" for km in KMER2_SYM_LIST],
)

KMER_MEAN_FREQ_TH = 1e-3

# Filter out low-frequency k-mers based on threshold
kmer_mean_freq = df_kmer2_sym.mean()
keep_kmers = kmer_mean_freq[kmer_mean_freq >= KMER_MEAN_FREQ_TH].index
df_kmer2_sym_filt = df_kmer2_sym[keep_kmers]


# ================================
# 8. Final Feature Matrix
# ================================

df_features_all = pd.concat(
    [
        df_pc_all,
        df_term_aa,
        df_term_group,
        df_term_charge,
        df_sw_aa,
        df_sw_group,
        df_sw_charge,
        df_kmer2_sym_filt,
    ],
    axis=1,
)

# ================================
# 9. Random Forest Model Training (High vs Low) on Full Dataset
# ================================

X = df_features_all.loc[df_binary.index]
y = df_binary["Class2"]

# Train the model on the entire dataset (X, y) without train/test splitting
clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X, y)

print("Model training completed on the entire dataset.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00
Model training completed on the entire dataset.


In [7]:
# ================================
# Prediction Function for New Data (Including All Features)
# ================================


def predict_from_fasta(fasta_path, trained_model, train_cols, kmer_cols):
    """Extracts features from a new FASTA file, predicts using the trained model,

    and returns a DataFrame containing ID, Prediction, Score_High, Sequence, and
    all calculated features.
    """

    # Internal cleaning function for non-standard amino acid substitution
    def extended_clean(seq):
        seq = seq.upper()
        seq = seq.replace("U", "C")
        seq = seq.replace("Z", "E")
        seq = seq.replace("B", "D")
        seq = seq.replace("J", "L")
        seq = seq.replace("O", "K")
        return seq

    # --- 1. Load Data ---

    new_seqs = {}
    for rec in SeqIO.parse(fasta_path, "fasta"):
        parts = rec.description.split("|")
        acc_id = parts[1] if len(parts) > 1 else rec.id
        new_seqs[acc_id] = str(rec.seq)

    df_new = pd.DataFrame(new_seqs.items(), columns=["ID", "Sequence"])

    # Apply amino acid substitution rules
    df_new["Sequence"] = df_new["Sequence"].map(extended_clean)

    # Exclude proteins containing unknown amino acid 'X'
    contains_x = df_new["Sequence"].str.contains("X")
    x_count = contains_x.sum()
    if x_count > 0:
        print(f"Info: Excluded {x_count} sequence(s) containing 'X'.")
        df_new = df_new[~contains_x].copy()

    if df_new.empty:
        print("Error: No valid sequences found (all sequences contained 'X').")
        return None

    # Reset index to ensure exact alignment during concatenation
    df_new = df_new.reset_index(drop=True)
    df_new["ID"] = df_new["ID"].astype(str)
    sequences = df_new["Sequence"].tolist()

    # --- 2. Feature Extraction (Strictly matched with training pipeline) ---

    # Basic physicochemical properties + AAC
    df_pc = pd.DataFrame(
        [physchem_features(s) for s in sequences], columns=pc_cols
    )

    # Terminal Bias
    df_t_aa = pd.DataFrame(
        [[termial_bias(s, a) for a in AAS] for s in sequences],
        columns=[f"TermBias_{a}" for a in AAS],
    )
    df_t_ge = pd.DataFrame(
        [[termial_bias(s, g) for g in AA_GROUPS_EXT.values()] for s in sequences],
        columns=[f"TermBias_{k}" for k in AA_GROUPS_EXT],
    )
    df_t_gc = pd.DataFrame(
        [[termial_bias(s, g) for g in AA_GROUPS.values()] for s in sequences],
        columns=[f"TermBias_{k}" for k in AA_GROUPS],
    )

    # Sliding Window
    df_s_aa = pd.DataFrame(
        [sliding_window_var_norm_aa(s, AAS) for s in sequences],
        columns=[f"SW{w}_Var_{a}" for w in SW_WINDOWS for a in AAS],
    )
    df_s_ge = pd.DataFrame(
        [sliding_window_var_norm_group(s, AA_GROUPS_EXT) for s in sequences],
        columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS_EXT],
    )
    df_s_gc = pd.DataFrame(
        [sliding_window_var_norm_group(s, AA_GROUPS) for s in sequences],
        columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS],
    )

    # k-mer (Retain and align only the columns kept during training: keep_kmers)
    df_k = pd.DataFrame(
        [kmer2_symmetric_features(s) for s in sequences],
        columns=[f"Pair_{km}" for km in KMER2_SYM_LIST],
    )
    df_k_filt = df_k.reindex(columns=kmer_cols, fill_value=0)

    # --- 3. Combine Final Feature Matrix ---
    df_features_new = pd.concat(
        [
            df_pc,
            df_t_aa,
            df_t_ge,
            df_t_gc,
            df_s_aa,
            df_s_ge,
            df_s_gc,
            df_k_filt,
        ],
        axis=1,
    )

    # Reorder columns to match the training feature alignment exactly
    df_features_new = df_features_new.reindex(columns=train_cols, fill_value=0)

    # --- 4. Model Prediction ---
    probs = trained_model.predict_proba(df_features_new)[:, 1]
    preds = trained_model.predict(df_features_new)

    # Base metadata and prediction columns
    df_meta = pd.DataFrame(
        {
            "ID": df_new["ID"],
            "Prediction": ["High" if p == 1 else "Low" for p in preds],
            "Score_High": probs,
            "Sequence": df_new["Sequence"],
        }
    )

    # --- 5. Concatenate Metadata with All Features ---
    df_res = pd.concat([df_meta, df_features_new], axis=1)

    return df_res


# ================================
# Execution
# ================================

new_fasta_file = "20251124_2820proteins_Low_group.fasta"

result_df = predict_from_fasta(
    fasta_path=new_fasta_file,
    trained_model=clf,
    train_cols=X.columns,
    kmer_cols=keep_kmers,
)

if result_df is not None:
    print("--- Prediction Results (Top 5) ---")
    print(result_df.head())

    print("\n--- Prediction Statistics ---")
    print(result_df["Prediction"].value_counts())

    # Export to CSV including all feature columns
    result_df.to_csv("prediction_output.csv", index=False)
    print("\nResults successfully saved to 'prediction_output.csv'.")

--- Prediction Results (Top 5) ---
       ID Prediction  Score_High  \
0  P31946        Low    0.066667   
1  P62258        Low    0.073333   
2  Q04917        Low    0.076667   
3  P61981        Low    0.080000   
4  P27348        Low    0.080000   

                                            Sequence          MW     GRAVY  \
0  MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...  28082.0763 -0.729268   
1  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...  29173.5752 -0.540000   
2  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...  28218.4010 -0.618293   
3  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...  28302.2676 -0.680162   
4  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...  27763.9374 -0.511837   

   Charge_7.5  Charge_6.8  DeltaCharge_6.8_7.5        pI  ...   Pair_ST  \
0  -14.828672  -13.997273             0.831399  4.764267  ...  0.008163   
1  -19.825038  -18.861724             0.963313  4.632514  ...  0.007874   
2  -13.892572  -13.139214             0.753358  4.7586